# Contradiction & Intervention

Two experiments:
1. **Observational**: Does the base model tolerate contradiction (Freud's primary process has no negation)?
2. **Interventional**: Can we decompose contradictory representations using nnsight?

The claim: the primary process (base model) holds contradictory drives in superposition.
The secondary process (SFT/DPO) introduces logical consistency by resolving toward one pole.

In [ ]:
import os, torch
os.chdir(os.path.expanduser('~/github/malign-logits'))

from malign_logits.psyche import Psyche

# Load from stash if available, otherwise load models
try:
    psyche = Psyche.from_family('olmo', load=False)
    # Test if logits are cached
    _ = psyche.primary_process.logits('test')
    print('Loaded from stash')
except:
    psyche = Psyche.from_family('olmo', load=True)
    print('Models loaded')

## Part 1: Observational — Contradiction tolerance across training stages

In [ ]:
results = psyche.contradiction_analysis()

import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'contested_words'} for r in results])
df

In [ ]:
from plotnine import *

MODEL_ORDER = ['base', 'sft', 'dpo', 'rlvr']
df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)

plot_df = df.melt(
    id_vars=['pair', 'model'],
    value_vars=['superposition', 'resolution'],
    var_name='measure', value_name='js'
)

(
    ggplot(plot_df, aes(x='model', y='js', fill='measure'))
    + geom_col(position='dodge', width=0.7, alpha=0.8)
    + facet_wrap('~pair', scales='free_y', ncol=3)
    + scale_fill_manual(values={'superposition': '#4e79a7', 'resolution': '#e15759'})
    + labs(
        title='Contradiction tolerance across alignment stages',
        subtitle='Superposition < resolution = primary process (tolerates contradiction)',
        x='Training stage', y='JS divergence',
    )
    + theme_minimal()
    + theme(
        figure_size=(12, 6),
        plot_background=element_rect(fill='white'),
        strip_text=element_text(size=11, weight='bold'),
    )
)

In [ ]:
# Ratio plot: < 1 = superposition, > 1 = resolution
(
    ggplot(df, aes(x='model', y='ratio', color='pair', group='pair'))
    + geom_line(size=1)
    + geom_point(size=3)
    + geom_hline(yintercept=1, linetype='dashed', color='#666666')
    + annotate('text', x=0.6, y=0.85, label='superposition\n(primary process)',
               color='#4e79a7', size=8, ha='left')
    + annotate('text', x=0.6, y=1.15, label='resolution\n(secondary process)',
               color='#e15759', size=8, ha='left')
    + labs(
        title='Contradiction ratio across alignment stages',
        subtitle='ratio = JS(AB, mean) / min(JS(AB, A), JS(AB, B))',
        x='Training stage', y='Superposition / Resolution ratio',
    )
    + theme_minimal()
    + theme(
        figure_size=(8, 5),
        plot_background=element_rect(fill='white'),
    )
)

In [ ]:
# Contested words — what does the model predict for the combined prompt?
for r in results:
    if r['pair'] == 'love/hate' and r['model'] == 'base':
        print(f"Pair: {r['pair']}  Model: {r['model'].upper()}")
        print(f"  A: {r['prompt_a']}")
        print(f"  B: {r['prompt_b']}")
        print(f"  AB: {r['prompt_ab']}")
        print(f"  Ratio: {r['ratio']:.3f}")
        print()
        for w in r['contested_words']:
            closer_to = 'MEAN' if abs(w['prob_ab'] - w['prob_mean']) < min(abs(w['prob_ab'] - w['prob_a']), abs(w['prob_ab'] - w['prob_b'])) else 'POLE'
            print(f"  {w['word']:15s}  A={w['prob_a']:.4f}  B={w['prob_b']:.4f}  AB={w['prob_ab']:.4f}  mean={w['prob_mean']:.4f}  → {closer_to}")

## Part 2: Interventional — Decomposing contradiction with nnsight

Can we identify a "love-hate" direction in the hidden states and manipulate it?

1. Extract hidden states for "She loved him" and "She hated him" at each layer
2. Compute the difference vector (love→hate direction)
3. Run "She loved and hated him" and intervene by adding/subtracting the direction
4. Measure whether the output shifts toward the corresponding pole

In [ ]:
from nnsight import LanguageModel
import torch

model = LanguageModel(
    'allenai/Olmo-3-1025-7B',
    device_map='mps',
    dtype=torch.float16,
    dispatch=True,
)
print(f'Loaded: {len(model.model.layers)} layers')

In [ ]:
prompt_a = "She loved him deeply and wanted to"
prompt_b = "She hated him deeply and wanted to"
prompt_ab = "She loved him and hated him and wanted to"

def get_hidden_states(model, prompt):
    """Extract last-position hidden state at each layer."""
    states = {}
    with model.trace(prompt) as tracer:
        for i, layer in enumerate(model.model.layers):
            states[i] = layer.output[0][0, -1, :].save()
    return {i: s.value.float().cpu() for i, s in states.items()}

print('Extracting hidden states...')
h_a = get_hidden_states(model, prompt_a)
h_b = get_hidden_states(model, prompt_b)
h_ab = get_hidden_states(model, prompt_ab)
print(f'Got {len(h_a)} layers, dim={h_a[0].shape[-1]}')

In [ ]:
# Compute love→hate direction at each layer
import torch.nn.functional as F

directions = {}
cosine_sims = []

for i in h_a:
    diff = h_b[i] - h_a[i]
    directions[i] = F.normalize(diff, dim=-1)
    
    # How much does AB project onto the love→hate axis?
    ab_centered = h_ab[i] - 0.5 * (h_a[i] + h_b[i])
    cos = F.cosine_similarity(ab_centered.unsqueeze(0), directions[i].unsqueeze(0)).item()
    cosine_sims.append({'layer': i, 'cosine_with_direction': cos})

cos_df = pd.DataFrame(cosine_sims)

(
    ggplot(cos_df, aes(x='layer', y='cosine_with_direction'))
    + geom_line(color='#4e79a7', size=1)
    + geom_point(color='#4e79a7', size=2)
    + geom_hline(yintercept=0, linetype='dashed', color='#666666')
    + labs(
        title='How much does "loved and hated" align with the love→hate direction?',
        subtitle='Cosine of (AB - midpoint) with (B - A) direction at each layer',
        x='Network layer', y='Cosine similarity',
    )
    + theme_minimal()
    + theme(figure_size=(10, 4), plot_background=element_rect(fill='white'))
)

In [ ]:
# Intervention: at a given layer, push AB toward A or toward B
# and measure the effect on the output distribution

from malign_logits.analysis import js_divergence

def intervene(model, prompt, layer_idx, direction, alpha=1.0):
    """Run prompt but add alpha * direction to hidden state at layer_idx.
    Returns logits at last position."""
    with model.trace(prompt) as tracer:
        h = model.model.layers[layer_idx].output[0]
        h[0, -1, :] += alpha * direction.to(h.device).half()
        model.model.layers[layer_idx].output = (h,) + model.model.layers[layer_idx].output[1:]
        logits = model.lm_head.output[0, -1, :].save()
    return logits.value.float().cpu()

# Get baseline logits for A, B, AB (no intervention)
def get_logits(model, prompt):
    with model.trace(prompt) as tracer:
        logits = model.lm_head.output[0, -1, :].save()
    return logits.value.float().cpu()

logits_a = get_logits(model, prompt_a)
logits_b = get_logits(model, prompt_b)
logits_ab = get_logits(model, prompt_ab)

print(f'JS(A, B) = {js_divergence(logits_a, logits_b):.4f}')
print(f'JS(AB, A) = {js_divergence(logits_ab, logits_a):.4f}')
print(f'JS(AB, B) = {js_divergence(logits_ab, logits_b):.4f}')

In [ ]:
# Intervene at multiple layers with varying strength
intervention_results = []

for layer_idx in [8, 16, 24, 28, 31]:
    direction = directions[layer_idx]
    # Scale by the norm of the actual difference (not unit vector)
    diff_norm = (h_b[layer_idx] - h_a[layer_idx]).norm().item()
    
    for alpha_frac in [-1.0, -0.5, 0, 0.5, 1.0]:
        alpha = alpha_frac * diff_norm
        logits_intervened = intervene(model, prompt_ab, layer_idx, direction, alpha)
        
        js_to_a = js_divergence(logits_intervened, logits_a)
        js_to_b = js_divergence(logits_intervened, logits_b)
        js_to_ab = js_divergence(logits_intervened, logits_ab)
        
        intervention_results.append({
            'layer': layer_idx,
            'alpha': alpha_frac,
            'js_to_A': js_to_a,
            'js_to_B': js_to_b,
            'js_to_AB': js_to_ab,
            'shift': js_to_a - js_to_b,  # positive = closer to B (hate)
        })
        print(f'  layer {layer_idx}, α={alpha_frac:+.1f}: JS→A={js_to_a:.4f} JS→B={js_to_b:.4f} shift={js_to_a - js_to_b:+.4f}')

int_df = pd.DataFrame(intervention_results)

In [ ]:
int_df['layer_str'] = 'layer ' + int_df['layer'].astype(str)

(
    ggplot(int_df, aes(x='alpha', y='shift', color='layer_str'))
    + geom_line(size=1)
    + geom_point(size=3)
    + geom_hline(yintercept=0, linetype='dashed', color='#666666')
    + labs(
        title='Intervention: pushing "loved and hated" along the love→hate axis',
        subtitle='shift = JS(intervened, A) - JS(intervened, B). Positive = closer to hate.',
        x='Intervention strength (α, fraction of ||B-A||)',
        y='Shift toward hate ←→ love',
        color='Layer',
    )
    + annotate('text', x=-0.9, y=-0.02, label='← pushed toward love', color='#999999', size=8)
    + annotate('text', x=0.5, y=0.02, label='pushed toward hate →', color='#999999', size=8)
    + theme_minimal()
    + theme(figure_size=(10, 5), plot_background=element_rect(fill='white'))
)

In [ ]:
# What words change most under intervention?
tokenizer = model.tokenizer

# Push AB strongly toward hate (alpha=1.0) at the most effective layer
best_layer = int_df.loc[int_df['alpha'] == 1.0].sort_values('shift', ascending=False).iloc[0]['layer']
direction = directions[int(best_layer)]
diff_norm = (h_b[int(best_layer)] - h_a[int(best_layer)]).norm().item()

logits_pushed = intervene(model, prompt_ab, int(best_layer), direction, diff_norm)

p_ab = torch.softmax(logits_ab, dim=-1)
p_pushed = torch.softmax(logits_pushed, dim=-1)
p_a = torch.softmax(logits_a, dim=-1)
p_b = torch.softmax(logits_b, dim=-1)

delta = p_pushed - p_ab
top_rising = delta.topk(10)
top_falling = (-delta).topk(10)

print(f'Intervention at layer {int(best_layer)}, pushing toward hate:')
print(f'\nWords RISING (boosted by hate direction):')
for v, idx in zip(top_rising.values, top_rising.indices):
    w = tokenizer.decode([idx]).strip()
    print(f'  {w:15s}  Δ={v.item():+.4f}  (AB={p_ab[idx]:.4f} → {p_pushed[idx]:.4f}, A={p_a[idx]:.4f}, B={p_b[idx]:.4f})')

print(f'\nWords FALLING (suppressed by hate direction):')
for v, idx in zip(top_falling.values, top_falling.indices):
    w = tokenizer.decode([idx]).strip()
    print(f'  {w:15s}  Δ={-v.item():+.4f}  (AB={p_ab[idx]:.4f} → {p_pushed[idx]:.4f}, A={p_a[idx]:.4f}, B={p_b[idx]:.4f})')

## Part 3: Does alignment change the geometry?

Repeat the intervention on the DPO model. If the contradiction vector is less effective
(smaller shift per unit alpha), alignment has made the contradiction space less linear —
the secondary process doesn't just suppress contradictions, it restructures how they're represented.

In [ ]:
model_dpo = LanguageModel(
    'allenai/Olmo-3-7B-Instruct-DPO',
    device_map='mps',
    dtype=torch.float16,
    dispatch=True,
)
print(f'DPO model loaded: {len(model_dpo.model.layers)} layers')

In [ ]:
h_a_dpo = get_hidden_states(model_dpo, prompt_a)
h_b_dpo = get_hidden_states(model_dpo, prompt_b)

dirs_dpo = {}
for i in h_a_dpo:
    diff = h_b_dpo[i] - h_a_dpo[i]
    dirs_dpo[i] = F.normalize(diff, dim=-1)

logits_a_dpo = get_logits(model_dpo, prompt_a)
logits_b_dpo = get_logits(model_dpo, prompt_b)
logits_ab_dpo = get_logits(model_dpo, prompt_ab)

dpo_results = []
for layer_idx in [8, 16, 24, 28, 31]:
    direction = dirs_dpo[layer_idx]
    diff_norm = (h_b_dpo[layer_idx] - h_a_dpo[layer_idx]).norm().item()
    
    for alpha_frac in [-1.0, -0.5, 0, 0.5, 1.0]:
        alpha = alpha_frac * diff_norm
        logits_int = intervene(model_dpo, prompt_ab, layer_idx, direction, alpha)
        js_a = js_divergence(logits_int, logits_a_dpo)
        js_b = js_divergence(logits_int, logits_b_dpo)
        dpo_results.append({
            'layer': layer_idx, 'alpha': alpha_frac,
            'shift': js_a - js_b, 'model_type': 'DPO',
        })

# Combine with base results
for r in intervention_results:
    r['model_type'] = 'BASE'

combined = pd.DataFrame(intervention_results + dpo_results)
combined['layer_str'] = 'layer ' + combined['layer'].astype(str)

(
    ggplot(combined[combined['layer'].isin([16, 28])],
           aes(x='alpha', y='shift', color='model_type', linetype='layer_str'))
    + geom_line(size=1)
    + geom_point(size=3)
    + geom_hline(yintercept=0, linetype='dashed', color='#666666')
    + scale_color_manual(values={'BASE': '#e15759', 'DPO': '#4e79a7'})
    + labs(
        title='Intervention effectiveness: BASE vs DPO',
        subtitle='Steeper slope = more linear contradiction representation',
        x='Intervention strength', y='Shift toward hate',
        color='Model', linetype='Layer',
    )
    + theme_minimal()
    + theme(figure_size=(10, 5), plot_background=element_rect(fill='white'))
)